# Retail Demand Forecasting

## 4. Time-Series Feature Engineering

This stage transforms the processed retail dataset into a forecasting-ready modelling dataset.

The feature engineering strategy focuses on temporal structure, historical demand behaviour, seasonality, promotional activity, holidays, and external economic information.

All historical demand features are constructed using information available prior to the prediction date to prevent target leakage.

## 4.2 Forecasting Objective

The forecasting problem is defined as predicting daily sales for each store and product family combination.

The primary forecasting unit is:

**Store × Product Family × Day**

The target variable is daily sales.

The modelling framework will initially focus on one-day-ahead forecasting, where information available up to day *t* is used to predict demand on day *t+1*.

This setup provides a clear foundation for comparing seasonal baselines with statistical and machine-learning forecasting models.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

## 4.3 Load Processed Dataset

The validated dataset produced during the preprocessing stage is loaded as the starting point for time-series feature engineering.

In [2]:
DATA_PATH = "data/processed_sales_data.csv"

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["date", "store_nbr", "family"]
).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")

Dataset shape: (3000888, 13)


## 4.4 Forecasting Data Structure

The dataset is validated to ensure that each observation represents a store-product-family combination on a specific date and that the target variable is available for the historical training period.

In [3]:
print("Date range:")
print(df["date"].min(), "to", df["date"].max())

print("\nUnique stores:", df["store_nbr"].nunique())

print("Unique product families:", df["family"].nunique())

print("Unique dates:", df["date"].nunique())

print("\nMissing sales values:", df["sales"].isna().sum())

Date range:
2013-01-01 00:00:00 to 2017-08-15 00:00:00

Unique stores: 54
Unique product families: 33
Unique dates: 1684

Missing sales values: 0


## 4.5 Forecasting Series Definition

Each unique combination of store and product family is treated as an individual demand series.

Historical observations are ordered chronologically within each series so that lagged and rolling features can be calculated without using future information.

## 4.6 Calendar Features

Calendar variables are created to capture recurring seasonal patterns in retail demand.

These features represent information that is known in advance for future dates and therefore do not introduce target leakage.

In [4]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["day_of_month"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [5]:
df[
    [
        "date",
        "year",
        "month",
        "quarter",
        "week_of_year",
        "day_of_month",
        "day_of_week",
        "is_weekend"
    ]
].head()

,date,year,month,quarter,week_of_year,day_of_month,day_of_week,is_weekend
0,2013-01-01,2013,1,1,1,1,1,0
1,2013-01-01,2013,1,1,1,1,1,0
2,2013-01-01,2013,1,1,1,1,1,0
3,2013-01-01,2013,1,1,1,1,1,0
4,2013-01-01,2013,1,1,1,1,1,0


## 4.7 Cyclical Seasonality Features

Cyclical transformations are used to represent recurring calendar patterns while preserving the circular relationship between the beginning and end of each seasonal cycle.

In [6]:
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df["day_of_week_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_of_week_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

df["week_sin"] = np.sin(
    2 * np.pi * df["week_of_year"] / 52
)

df["week_cos"] = np.cos(
    2 * np.pi * df["week_of_year"] / 52
)

## 4.8 Historical Demand Features

Lagged sales variables capture recent demand behaviour for each store-product family series.

The lag structure includes daily, weekly, and multi-week historical demand.

All lagged values are shifted so that the target day's sales are never used as an input feature.

In [7]:
group_cols = ["store_nbr", "family"]

for lag in [1, 2, 3, 7, 14, 28]:
    df[f"sales_lag_{lag}"] = (
        df.groupby(group_cols)["sales"]
        .shift(lag)
    )

In [8]:
df[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "sales_lag_1",
        "sales_lag_7",
        "sales_lag_14",
        "sales_lag_28"
    ]
].head(20)

,date,store_nbr,family,sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28
0,2013-01-01,1,AUTOMOTIVE,0.00,NaN,NaN,NaN,NaN
1,2013-01-01,1,BABY CARE,0.00,NaN,NaN,NaN,NaN
2,2013-01-01,1,BEAUTY,0.00,NaN,NaN,NaN,NaN
3,2013-01-01,1,BEVERAGES,0.00,NaN,NaN,NaN,NaN
4,2013-01-01,1,BOOKS,0.00,NaN,NaN,NaN,NaN
5,2013-01-01,1,BREAD/BAKERY,0.00,NaN,NaN,NaN,NaN
6,2013-01-01,1,CELEBRATION,0.00,NaN,NaN,NaN,NaN
7,2013-01-01,1,CLEANING,0.00,NaN,NaN,NaN,NaN
8,2013-01-01,1,DAIRY,0.00,NaN,NaN,NaN,NaN
9,2013-01-01,1,DELI,0.00,NaN,NaN,NaN,NaN


## 4.9 Rolling Demand Features

Rolling statistics summarize recent historical demand and provide the models with information about local demand level and variability.

A one-period shift is applied before calculating rolling statistics to ensure that the target day's sales are excluded from the feature calculation.

In [9]:
sales_shifted = (
    df.groupby(group_cols)["sales"]
    .shift(1)
)

for window in [7, 14, 28]:
    df[f"sales_rolling_mean_{window}"] = (
        sales_shifted
        .groupby(
            [
                df["store_nbr"],
                df["family"]
            ]
        )
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

    df[f"sales_rolling_std_{window}"] = (
        sales_shifted
        .groupby(
            [
                df["store_nbr"],
                df["family"]
            ]
        )
        .transform(
            lambda x: x.rolling(window).std()
        )
    )

In [10]:
df[
    [
        "sales",
        "sales_lag_7",
        "sales_rolling_mean_7",
        "sales_rolling_std_7",
        "sales_rolling_mean_28",
        "sales_rolling_std_28"
    ]
].head(30)

,sales,sales_lag_7,sales_rolling_mean_7,sales_rolling_std_7,sales_rolling_mean_28,sales_rolling_std_28
0,0.00,NaN,NaN,NaN,NaN,NaN
1,0.00,NaN,NaN,NaN,NaN,NaN
2,0.00,NaN,NaN,NaN,NaN,NaN
3,0.00,NaN,NaN,NaN,NaN,NaN
4,0.00,NaN,NaN,NaN,NaN,NaN
5,0.00,NaN,NaN,NaN,NaN,NaN
6,0.00,NaN,NaN,NaN,NaN,NaN
7,0.00,NaN,NaN,NaN,NaN,NaN
8,0.00,NaN,NaN,NaN,NaN,NaN
9,0.00,NaN,NaN,NaN,NaN,NaN


## 4.10 Demand Momentum

Recent demand momentum is represented using ratios between short-term and longer-term rolling averages.

Values above one indicate that recent demand is higher than the longer historical average, while values below one indicate weaker recent demand.

In [11]:
df["sales_momentum_7_28"] = (
    df["sales_rolling_mean_7"]
    / df["sales_rolling_mean_28"]
)

df["sales_momentum_7_28"] = (
    df["sales_momentum_7_28"]
    .replace([np.inf, -np.inf], np.nan)
)

## 4.11 Promotion Features

Promotional activity is incorporated as an explanatory variable because promotion availability can influence retail demand.

Current-day promotion information is retained as a potential predictor because promotion schedules are available as part of the forecasting dataset.

In [12]:
df["onpromotion"].describe()

count   3,000,888.00
mean            2.60
std            12.22
min             0.00
25%             0.00
50%             0.00
75%             0.00
max           741.00
Name: onpromotion, dtype: float64

In [13]:
df["onpromotion"] = pd.to_numeric(
    df["onpromotion"],
    errors="coerce"
)

df["promotion_available"] = (
    df["onpromotion"] > 0
).astype(int)

In [14]:
df["promotion_lag_1"] = (
    df.groupby(group_cols)["onpromotion"]
    .shift(1)
)

df["promotion_rolling_7"] = (
    df.groupby(group_cols)["onpromotion"]
    .transform(
        lambda x: x.shift(1).rolling(7).mean()
    )
)

In [15]:
"is_holiday" in df.columns

True

In [16]:
df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

## 4.12 External Economic Features

External economic information is incorporated to evaluate whether broader economic conditions provide additional predictive information beyond historical demand and calendar variables.

In [17]:
if "dcoilwtico" in df.columns:
    df["oil_price_lag_1"] = (
        df.groupby(group_cols)["dcoilwtico"]
        .shift(1)
    )

    df["oil_price_change"] = (
        df["dcoilwtico"]
        .pct_change()
    )

## 4.13 Historical Window Validation

The forecasting features require sufficient historical observations.

Rows without the minimum historical window required for the selected lag and rolling features are removed from the modelling dataset.

In [18]:
required_history = [
    "sales_lag_28",
    "sales_rolling_mean_28",
    "sales_rolling_std_28"
]

before_rows = len(df)

df_model = df.dropna(
    subset=required_history
).copy()

after_rows = len(df_model)

print(f"Rows before historical filtering: {before_rows:,}")
print(f"Rows after historical filtering: {after_rows:,}")
print(f"Rows removed: {before_rows - after_rows:,}")

Rows before historical filtering: 3,000,888
Rows after historical filtering: 2,950,992
Rows removed: 49,896


In [19]:
feature_missingness = (
    df_model.isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missingness[
    feature_missingness > 0
]

dcoilwtico             910602
oil_price_lag_1        910602
sales_momentum_7_28    659078
transactions           236907
dtype: int64

## 4.14 Chronological Data Split

A chronological split is used to preserve the temporal structure of the forecasting problem.

The model is trained only on historical observations and evaluated on later periods.

Random train-test splitting is avoided because it could allow future demand patterns to influence model training.

In [20]:
print("Start:", df_model["date"].min())
print("End:", df_model["date"].max())

print("\nYear distribution:")
print(df_model["year"].value_counts().sort_index())

Start: 2013-01-29 00:00:00
End: 2017-08-15 00:00:00

Year distribution:
year
2013    598752
2014    648648
2015    648648
2016    650430
2017    404514
Name: count, dtype: int64


## 4.15 Forecasting Feature Set

The modelling dataset combines historical demand, calendar seasonality, promotion information, and external economic signals.

The target variable remains daily sales.

In [21]:
target = "sales"

feature_columns = [
    "store_nbr",
    "family",
    "year",
    "month",
    "quarter",
    "week_of_year",
    "day_of_month",
    "day_of_week",
    "is_weekend",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "week_sin",
    "week_cos",
    "sales_lag_1",
    "sales_lag_2",
    "sales_lag_3",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14",
    "sales_rolling_mean_28",
    "sales_rolling_std_7",
    "sales_rolling_std_14",
    "sales_rolling_std_28",
    "sales_momentum_7_28",
    "onpromotion",
    "promotion_available",
    "promotion_lag_1",
    "promotion_rolling_7"
]

if "dcoilwtico" in df_model.columns:
    feature_columns.extend([
        "dcoilwtico",
        "oil_price_lag_1",
        "oil_price_change"
    ])

if "is_holiday" in df_model.columns:
    feature_columns.append("is_holiday")

feature_columns = [
    col for col in feature_columns
    if col in df_model.columns
]

print(f"Number of modelling features: {len(feature_columns)}")
print("\nFeatures:")
for feature in feature_columns:
    print(feature)

Number of modelling features: 36

Features:
store_nbr
family
year
month
quarter
week_of_year
day_of_month
day_of_week
is_weekend
month_sin
month_cos
day_of_week_sin
day_of_week_cos
week_sin
week_cos
sales_lag_1
sales_lag_2
sales_lag_3
sales_lag_7
sales_lag_14
sales_lag_28
sales_rolling_mean_7
sales_rolling_mean_14
sales_rolling_mean_28
sales_rolling_std_7
sales_rolling_std_14
sales_rolling_std_28
sales_momentum_7_28
onpromotion
promotion_available
promotion_lag_1
promotion_rolling_7
dcoilwtico
oil_price_lag_1
oil_price_change
is_holiday


In [22]:
model_data = df_model[
    ["date", "store_nbr", "family", "sales"] + feature_columns
].copy()

print("Final modelling dataset shape:")
print(model_data.shape)

Final modelling dataset shape:
(2950992, 40)


In [23]:
model_data.head()

,date,store_nbr,family,sales,store_nbr,family,year,month,quarter,week_of_year,day_of_month,day_of_week,is_weekend,month_sin,month_cos,day_of_week_sin,day_of_week_cos,week_sin,week_cos,sales_lag_1,sales_lag_2,sales_lag_3,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,sales_rolling_std_14,sales_rolling_std_28,sales_momentum_7_28,onpromotion,promotion_available,promotion_lag_1,promotion_rolling_7,dcoilwtico,oil_price_lag_1,oil_price_change,is_holiday
49896,2013-01-29,1,AUTOMOTIVE,2.00,1,AUTOMOTIVE,2013,1,1,5,29,1,0,0.50,0.87,0.78,0.62,0.57,0.82,3.00,2.00,4.00,1.00,1.00,0.00,2.57,2.14,2.14,1.72,1.70,1.46,1.20,0,0,0.00,0.00,97.62,95.95,0.02,0
49897,2013-01-29,1,BABY CARE,0.00,1,BABY CARE,2013,1,1,5,29,1,0,0.50,0.87,0.78,0.62,0.57,0.82,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,0,0,0.00,0.00,97.62,95.95,0.00,0
49898,2013-01-29,1,BEAUTY,3.00,1,BEAUTY,2013,1,1,5,29,1,0,0.50,0.87,0.78,0.62,0.57,0.82,0.00,0.00,1.00,1.00,0.00,0.00,0.57,1.21,1.32,0.79,1.42,1.33,0.43,0,0,0.00,0.00,97.62,95.95,0.00,0
49899,2013-01-29,1,BEVERAGES,932.00,1,BEVERAGES,2013,1,1,5,29,1,0,0.50,0.87,0.78,0.62,0.57,0.82,772.00,417.00,965.00,"1,037.00","1,149.00",0.00,884.57,946.36,925.89,237.43,252.83,300.12,0.96,0,0,0.00,0.00,97.62,95.95,0.00,0
49900,2013-01-29,1,BOOKS,0.00,1,BOOKS,2013,1,1,5,29,1,0,0.50,0.87,0.78,0.62,0.57,0.82,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,0,0,0.00,0.00,97.62,95.95,0.00,0


## 4.16 Export Feature-Engineered Dataset

The final feature-engineered dataset is exported for use in the baseline and machine-learning forecasting stages.

In [25]:
os.makedirs("outputs/tables", exist_ok=True)

FEATURE_DATA_PATH = (
    "outputs/tables/forecasting_features.csv"
)

model_data.to_csv(
    FEATURE_DATA_PATH,
    index=False
)

print(
    f"Feature-engineered dataset saved to: "
    f"{FEATURE_DATA_PATH}"
)

Feature-engineered dataset saved to: outputs/tables/forecasting_features.csv


## 4.17 Feature Engineering Summary

The forecasting dataset now contains:

- Calendar and seasonal features
- Cyclical representations of recurring time patterns
- Historical sales lags
- Rolling demand statistics
- Demand momentum indicators
- Promotion-related variables
- Holiday information where available
- External economic information
- Store and product-family identifiers

Historical demand features were constructed using shifted observations to prevent the target day's sales from entering its own predictors.

The resulting dataset provides the foundation for chronological baseline evaluation and machine-learning forecasting.